<div style="background:linear-gradient(135deg,#117A65 0%,#1E8449 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#A9DFBF;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 3 · CLASE 8</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: LDA y Análisis Discriminante</h1>
  <p style="color:#A9DFBF;font-size:16px;margin:0 0 24px 0;font-style:italic;">Fisher · Scatter matrices · Proyección discriminante</p>
  <hr style="border-color:#52BE80;margin:0 0 20px 0;">
  <p style="color:#EAFAF1;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('Setup listo')

---
## 🟢 Ejercicio 1 — Scatter matrices de Fisher

1. Calcular Sᴮ y Sᵥᵥ manualmente para los datos dados
2. Calcular la dirección de Fisher w = Sᵥᵥ⁻¹(μ₁ − μ₀)
3. Proyectar los datos sobre w y graficar histogramas
4. Calcular la frontera de decisión y el accuracy manual

In [ ]:
np.random.seed(SEED)
n=120
X0_e1=np.random.multivariate_normal([1,4],[[1.5,0.4],[0.4,1.0]],n)
X1_e1=np.random.multivariate_normal([5,1],[[1.0,-0.3],[-0.3,1.5]],n)
X_e1=np.vstack([X0_e1,X1_e1]); y_e1=np.array([0]*n+[1]*n)
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n=120
X0_e1=np.random.multivariate_normal([1,4],[[1.5,0.4],[0.4,1.0]],n)
X1_e1=np.random.multivariate_normal([5,1],[[1.0,-0.3],[-0.3,1.5]],n)
X_e1=np.vstack([X0_e1,X1_e1]); y_e1=np.array([0]*n+[1]*n)
mu0,mu1=X0_e1.mean(0),X1_e1.mean(0)
Sw_e=(X0_e1-mu0).T@(X0_e1-mu0)+(X1_e1-mu1).T@(X1_e1-mu1)
Sb_e=n*np.outer(mu1-mu0,mu1-mu0)
w_e=np.linalg.inv(Sw_e)@(mu1-mu0); w_e/=np.linalg.norm(w_e)
z0_e=X0_e1@w_e; z1_e=X1_e1@w_e; th_e=(z0_e.mean()+z1_e.mean())/2
y_pred_e=((X_e1@w_e)>=th_e).astype(int)
print(f'w Fisher: {w_e.round(4)}')
print(f'Frontera: {th_e:.4f}')
print(f'Accuracy manual: {(y_pred_e==y_e1).mean():.4f}')
lda_e1=LinearDiscriminantAnalysis(); lda_e1.fit(X_e1,y_e1)
print(f'Accuracy sklearn: {accuracy_score(y_e1,lda_e1.predict(X_e1)):.4f}')
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(X0_e1[:,0],X0_e1[:,1],s=20,alpha=0.5,color='#E74C3C',label='C0')
axes[0].scatter(X1_e1[:,0],X1_e1[:,1],s=20,alpha=0.5,color='#1E8449',label='C1')
c=X_e1.mean(0); axes[0].annotate('',xy=c+2.5*w_e,xytext=c-2.5*w_e,arrowprops=dict(arrowstyle='<->',color='#117A65',lw=2))
axes[0].set(title='Fisher — dirección'); axes[0].legend(); axes[0].grid(True,alpha=0.2)
axes[1].hist(z0_e,bins=25,alpha=0.6,color='#E74C3C',density=True,label='C0')
axes[1].hist(z1_e,bins=25,alpha=0.6,color='#1E8449',density=True,label='C1')
axes[1].axvline(th_e,color='#117A65',lw=2,linestyle='--',label=f'Frontera={th_e:.2f}')
axes[1].set(title='Proyecciones'); axes[1].legend(); axes[1].grid(True,alpha=0.2)
plt.tight_layout(); plt.show()

---
## 🟡 Ejercicio 2 — LDA vs QDA: cuándo difieren

Generar datos con covarianzas distintas por clase. Comparar LDA vs QDA vs Logística.
¿Cuál modelo gana? ¿Por qué?

In [ ]:
np.random.seed(SEED); n2=200
X0_e2=np.random.multivariate_normal([0,0],[[3,0.5],[0.5,1]],n2)
X1_e2=np.random.multivariate_normal([3,3],[[0.5,0],[0,4]],n2)  # Sigma muy distinta
X_e2=np.vstack([X0_e2,X1_e2]); y_e2=np.array([0]*n2+[1]*n2)
Xtr2,Xte2,ytr2,yte2=train_test_split(X_e2,y_e2,test_size=0.25,random_state=SEED)
# --- Comparar LDA, QDA y Logística ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n2=200
X0_e2=np.random.multivariate_normal([0,0],[[3,0.5],[0.5,1]],n2)
X1_e2=np.random.multivariate_normal([3,3],[[0.5,0],[0,4]],n2)
X_e2=np.vstack([X0_e2,X1_e2]); y_e2=np.array([0]*n2+[1]*n2)
Xtr2,Xte2,ytr2,yte2=train_test_split(X_e2,y_e2,test_size=0.25,random_state=SEED)
print(f'{"Modelo":12s} {"AUC":>8s} {"Acc":>8s}')
print('─'*30)
for nm,m in [('LDA',LinearDiscriminantAnalysis()),
             ('QDA',QuadraticDiscriminantAnalysis()),
             ('Logistica',LogisticRegression(C=1e6,solver='lbfgs',max_iter=500))]:
    m.fit(Xtr2,ytr2)
    auc=roc_auc_score(yte2,m.predict_proba(Xte2)[:,1])
    acc=accuracy_score(yte2,m.predict(Xte2))
    print(f'  {nm:10s} {auc:>8.4f} {acc:>8.4f}')
print('\nQDA deberia superar a LDA cuando las covarianzas son muy distintas.')

---
## 🟡 Ejercicio 3 — LDA multiclase y proyección

Con 3 clases de clientes bancarios:
1. Ajustar LDA y reportar accuracy
2. Proyectar en espacio LD1-LD2 y graficar
3. Identificar qué features discriminan más (coeficientes LD1)
4. Predecir segmento para 3 nuevos clientes

In [ ]:
np.random.seed(SEED); n3=150
S_com=np.array([[2,0.5,0],[0.5,2,0.3],[0,0.3,1]])
X_e3=np.vstack([np.random.multivariate_normal(mu,S_com,n3) for mu in [[-2,-2,0],[0,2,1],[3,-1,-1]]])
y_e3=np.repeat(['Riesgo_Alto','Riesgo_Medio','Riesgo_Bajo'],n3)
feats_e3=['deuda','historial','ingresos']
nuevos_e3=np.array([[2.5,-1,0.5],[-0.5,1.8,-0.5],[1.0,0.5,0.8]])
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n3=150
S_com=np.array([[2,0.5,0],[0.5,2,0.3],[0,0.3,1]])
X_e3=np.vstack([np.random.multivariate_normal(mu,S_com,n3) for mu in [[-2,-2,0],[0,2,1],[3,-1,-1]]])
y_e3=np.repeat(['Riesgo_Alto','Riesgo_Medio','Riesgo_Bajo'],n3)
feats_e3=['deuda','historial','ingresos']
nuevos_e3=np.array([[2.5,-1,0.5],[-0.5,1.8,-0.5],[1.0,0.5,0.8]])
Xtr3,Xte3,ytr3,yte3=train_test_split(X_e3,y_e3,test_size=0.25,random_state=SEED)
lda3=LinearDiscriminantAnalysis(); lda3.fit(Xtr3,ytr3)
acc3=accuracy_score(yte3,lda3.predict(Xte3))
print(f'Accuracy: {acc3:.4f}')
print(classification_report(yte3,lda3.predict(Xte3)))
X_proj3=lda3.transform(Xte3)
fig,ax=plt.subplots(figsize=(7,5))
for cls,color in zip(['Riesgo_Alto','Riesgo_Bajo','Riesgo_Medio'],['#E74C3C','#1E8449','#F39C12']):
    m=np.array(yte3)==cls
    ax.scatter(X_proj3[m,0],X_proj3[m,1],s=25,alpha=0.7,color=color,label=cls)
ax.set(xlabel='LD1',ylabel='LD2',title=f'Proyeccion LDA (Acc={acc3:.3f})')
ax.legend(); ax.grid(True,alpha=0.2); plt.tight_layout(); plt.show()
print('\nCoeficientes LD1 (importancia):')
for nm,c in zip(feats_e3,lda3.scalings_[:,0]):
    print(f'  {nm}: {c:+.4f}')
preds3=lda3.predict(nuevos_e3); probs3=lda3.predict_proba(nuevos_e3)
print('\nNuevos clientes:')
for i,(p,pr) in enumerate(zip(preds3,probs3)):
    print(f'  Cliente {chr(65+i)}: {p}  P={dict(zip(lda3.classes_,pr.round(3)))}')

---
## 🔴 Ejercicio 4 — Pipeline completo de clasificación

Implementar `discriminant_pipeline(X_raw, y, feats, nuevos=None)` que:
1. Estandarice y ajuste LDA y QDA
2. Compare ambos por accuracy y AUC
3. Elija el mejor y reporte coeficientes
4. Clasifique nuevos casos con probabilidades

In [ ]:
def discriminant_pipeline(X_raw, y, feats, nuevos=None):
    pass

np.random.seed(SEED); n4=800
Sigma4=np.eye(5)*2+0.5*np.ones((5,5))
X_e4=np.vstack([np.random.multivariate_normal(mu,Sigma4,n4//4) for mu in [[-2,-2,0,1,-1],[0,0,1,-1,0],[2,2,-1,0,1],[1,-2,2,1,-1]]])
y_e4=np.repeat(['Premium','Estandar','Basico','Riesgo'],n4//4)
feats_e4=['f1','f2','f3','f4','f5']
nuevos_e4=np.array([[-1.5,-1.5,0,0.5,-0.5],[1.5,1.5,-0.5,0,0.5]])
discriminant_pipeline(X_e4,y_e4,feats_e4,nuevos_e4)

In [ ]:
# ✅ SOLUCIÓN
def discriminant_pipeline(X_raw,y,feats,nuevos=None):
    sc=StandardScaler(); Xsc=sc.fit_transform(X_raw)
    Xtr,Xte,ytr,yte=train_test_split(Xsc,y,test_size=0.2,random_state=SEED)
    resultados={}
    for nm,m in [('LDA',LinearDiscriminantAnalysis()),('QDA',QuadraticDiscriminantAnalysis())]:
        m.fit(Xtr,ytr); acc=accuracy_score(yte,m.predict(Xte))
        resultados[nm]=(m,acc); print(f'  {nm}: Acc={acc:.4f}')
    mejor_nm=max(resultados,key=lambda k:resultados[k][1])
    mejor_m=resultados[mejor_nm][0]
    print(f'\nMejor: {mejor_nm}')
    print(classification_report(yte,mejor_m.predict(Xte)))
    if hasattr(mejor_m,'scalings_'):
        print('Coeficientes LD1:')
        for nm,c in zip(feats,mejor_m.scalings_[:,0]):
            print(f'  {nm}: {c:+.4f}')
    if nuevos is not None:
        nuevos_sc=sc.transform(nuevos)
        preds=mejor_m.predict(nuevos_sc); probs=mejor_m.predict_proba(nuevos_sc)
        print('\nNuevos casos:')
        for i,(p,pr) in enumerate(zip(preds,probs)):
            print(f'  Caso {i+1}: {p}  P={dict(zip(mejor_m.classes_,pr.round(3)))}')

np.random.seed(SEED); n4=800
Sigma4=np.eye(5)*2+0.5*np.ones((5,5))
X_e4=np.vstack([np.random.multivariate_normal(mu,Sigma4,n4//4) for mu in [[-2,-2,0,1,-1],[0,0,1,-1,0],[2,2,-1,0,1],[1,-2,2,1,-1]]])
y_e4=np.repeat(['Premium','Estandar','Basico','Riesgo'],n4//4)
feats_e4=['f1','f2','f3','f4','f5']
nuevos_e4=np.array([[-1.5,-1.5,0,0.5,-0.5],[1.5,1.5,-0.5,0,0.5]])
discriminant_pipeline(X_e4,y_e4,feats_e4,nuevos_e4)

---
<div style="background:#117A65;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Clase 9</strong><br>
Scores discriminantes · QDA · Caso marketing completo · Fin Módulo 3<br>
<em>Docente: Josef Rodriguez · Curso 8 · Módulo 3</em>
</div>